# TerraTree — Step 1: Data Acquisition (Sundarbans)
Study area: **Sundarbans National Park / Tiger Reserve, West Bengal, India**

This notebook:
1. Authenticates Earth Engine
2. Loads the Sundarbans boundary (via OpenStreetMap — this pattern is confirmed working)
3. Pulls a multi-temporal Sentinel-2 (optical) collection, cloud-masked
4. Pulls a multi-temporal Sentinel-1 (SAR) collection, VV/VH only, correctly typed for export
5. Loads Global Mangrove Watch as a reference/training layer
6. Visualizes everything on an interactive map
7. Exports clipped composites + reference layer to Google Drive for notebook 02

Run cells top to bottom. First run will ask you to authenticate — follow the link, sign in, paste the code.

In [ ]:
!pip install geemap -q

In [ ]:
import ee
import geemap

GEE_PROJECT_ID = "terratree"

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT_ID)

## Define the study area

We fetch the boundary from OpenStreetMap via Nominatim — this is the pattern
that worked reliably (WDPA's name/spatial search came up empty for our
previous area, OSM did not). If this specific name doesn't resolve, the
printed response will show what OSM does have — adjust the query string
and retry rather than falling back to a guessed bounding box.

In [ ]:
import requests

resp = requests.get(
    "https://nominatim.openstreetmap.org/search",
    params={
        "q": "Sundarbans National Park, West Bengal, India",
        "format": "geojson",
        "polygon_geojson": 1,
        "limit": 1
    },
    headers={"User-Agent": "terratree-major-project"}
)
data = resp.json()

if data.get('features'):
    aoi = ee.Geometry(data['features'][0]['geometry'])
    print("Loaded Sundarbans boundary from OpenStreetMap.")
else:
    print("OSM lookup failed — inspect `data` below and adjust the query.")
    print(data)
    # Rough fallback bounding box covering the Indian Sundarbans, verify on map before trusting
    aoi = ee.Geometry.Rectangle([88.75, 21.50, 89.20, 22.20])

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, 10)
Map.addLayer(aoi, {'color': 'red'}, 'Study area (Sundarbans)')
Map

## Global Mangrove Watch — reference/training layer

This is what replaces field ground-truth. GMW gives verified mangrove
extent for multiple years (1996–2020), hosted as a community Earth Engine
asset. If this specific asset ever fails to load (permissions do
occasionally change on community-hosted assets), the fallback is Google's
own first-party `LANDSAT/MANGROVE_FORESTS` dataset (year 2000 only, no
multi-year change, but always available).

In [ ]:
try:
    gmw = ee.ImageCollection(
        "projects/sat-io/open-datasets/GMW/extent/GMW_V3"
    )
    gmw_latest = gmw.sort('system:time_start', False).first().clip(aoi)
    gmw_source = "GMW_V3 (multi-year, community catalog)"
except Exception as e:
    print("GMW_V3 failed to load, falling back to LANDSAT/MANGROVE_FORESTS:", e)
    gmw_latest = ee.ImageCollection('LANDSAT/MANGROVE_FORESTS').mosaic().clip(aoi)
    gmw_source = "LANDSAT/MANGROVE_FORESTS (year 2000 baseline)"

print("Using reference layer:", gmw_source)

Map.addLayer(gmw_latest, {'palette': ['00FF00'], 'min': 0, 'max': 1}, 'Mangrove reference (GMW)')
Map

## Sentinel-2 optical collection (cloud-masked, multi-temporal)

Sundarbans is monsoon-affected and persistently cloudy for parts of the
year — expect a lower cloud-free scene count than a drier region. If the
scene count below comes back very low, widen `MAX_CLOUD_PCT` or the date
range rather than assuming something is broken.

In [ ]:
START_DATE = '2022-01-01'
END_DATE = '2024-12-31'
MAX_CLOUD_PCT = 30

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000).copyProperties(image, ['system:time_start'])

s2_collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
    .map(mask_s2_clouds)
)

print('Sentinel-2 scenes matching filters:', s2_collection.size().getInfo())

In [ ]:
s2_median = s2_collection.median().clip(aoi)

vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}
Map.addLayer(s2_median, vis_params, 'Sentinel-2 median composite (true color)')
Map

## Sentinel-1 SAR collection (VV + VH, IW mode)

SAR sees through the cloud cover that limits optical usability here —
especially valuable given how persistently cloudy this region is.

Note: we explicitly select only VV/VH bands and cast to Float before
exporting. This is a fix carried over from the earlier build — including
the `angle` band or leaving mixed dtypes in the composite caused an
export failure ("inconsistent types: Float64 and Float32") last time.

In [ ]:
s1_collection = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
)

print('Sentinel-1 scenes matching filters:', s1_collection.size().getInfo())

In [ ]:
s1_median = (
    s1_collection
    .select(['VV', 'VH'])
    .median()
    .toFloat()
    .clip(aoi)
)

vis_params_s1 = {'bands': ['VV'], 'min': -20, 'max': 0}
Map.addLayer(s1_median, vis_params_s1, 'Sentinel-1 VV median composite')
Map

## Export to Google Drive

Exports run asynchronously (check progress via the polling snippet below,
not the Code Editor Tasks page — that UI can lag/cache and show stale
project state).

In [ ]:
export_s2 = ee.batch.Export.image.toDrive(
    image=s2_median,
    description='sundarbans_s2_median_composite',
    folder='terratree',
    fileNamePrefix='sundarbans_s2_median',
    region=aoi,
    scale=10,
    maxPixels=1e13
)
export_s2.start()

export_s1 = ee.batch.Export.image.toDrive(
    image=s1_median,
    description='sundarbans_s1_median_composite',
    folder='terratree',
    fileNamePrefix='sundarbans_s1_median',
    region=aoi,
    scale=10,
    maxPixels=1e13
)
export_s1.start()

export_gmw = ee.batch.Export.image.toDrive(
    image=gmw_latest.toFloat(),
    description='sundarbans_gmw_reference',
    folder='terratree',
    fileNamePrefix='sundarbans_gmw_reference',
    region=aoi,
    scale=10,
    maxPixels=1e13
)
export_gmw.start()

print('Export tasks started for S2, S1, and the GMW reference layer.')

In [ ]:
import time

tasks = [export_s2, export_s1, export_gmw]
done_states = {'COMPLETED', 'FAILED', 'CANCELLED'}

while True:
    states = [t.status()['state'] for t in tasks]
    print(states)
    if all(s in done_states for s in states):
        break
    time.sleep(30)

print("All exports finished. Check the 'terratree' folder in Google Drive.")

## Next steps
- [ ] Confirm the AOI boundary actually covers the Sundarbans reserve (zoom in on the map above)
- [ ] Confirm the GMW reference layer overlays sensibly on the true-color composite
- [ ] Once exports finish, move to `02_feature_extraction.ipynb`
- [ ] Finalize classification classes: mangrove / non-mangrove vegetation / water-mudflat as the baseline; refine further if time allows